## FOBOS I2C Test

7/30/2024,  Jens-Peter Kaps

In [41]:
import time
from pynq import Overlay
from pynq import Clocks
from pynq import MMIO

In [42]:
from pynq_drivers.config.pynq_conf import IP, PORT, OVERLAY_FILE, FOBOS_HOME

In [43]:
ol = Overlay(OVERLAY_FILE)

after overlay is loaded, set clock back to 100 MHZ

In [44]:
Clocks.fclk0_mhz=100
print(Clocks.fclk0_mhz)

100.0


In [45]:
dutcomm = ol.dutcomm_0

In [46]:
dutcomm_STATUS = 0x04

In [47]:
dutcomm.read(dutcomm_STATUS)

2

In [48]:
from pynq.lib.iic import AxiIIC

In [49]:
iic = ol.axi_iic_0

In [50]:
iic=AxiIIC(ol.ip_dict["axi_iic_0"])

In [62]:
# DUT Controller Registers
dut_select     = 0x38
dut_reset      = 0x1C

# DUT Select Settings
dut_FOBOS      = 0x00
dut_CW305      = 0x01
dut_I2C        = 0x02
dut_serial     = 0x03

In [52]:
dutctrl_addr = ol.ip_dict["dut_controller_0"]["phys_addr"]
dutctrl_addr_range = ol.ip_dict["dut_controller_0"]["addr_range"]
dutctrl = MMIO(dutctrl_addr, dutctrl_addr_range)
dutctrl.device

In [13]:
dutctrl.write(dut_select,dut_I2C)

In [14]:
dutctrl.read(dut_select)

2

In [15]:
test_vector="00C00010AC0A7F8C2FAAC49775A616B7C0CC21D800C100100123456789ABCDEF00112233445566770081001000800001"

In [16]:
#test_vector="00C00010AC" #AC0A7F8C" #2FAAC49775A616B7C0CC21D8"

In [17]:
data = bytearray.fromhex(test_vector)

In [18]:
data = list(data)

In [19]:
#data

In [27]:
iic.wait()

In [30]:
iic.send(0x40,data,len(data),0)

48

In [23]:
while(dutcomm.read(dutcomm_STATUS) & 0x100):
    time.sleep(0.1)

In [237]:
# rx_data = []

In [238]:
# iic.receive(0x40,rx_data,2,0)

In [239]:
# rx_data

In [31]:
c_data=AxiIIC._ffi.new("unsigned char[16]")

In [32]:
iic.receive(0x40,c_data,16,0)

16

In [55]:
iic.register_map

RegisterMap {
  GIE = Register(GIE=0),
  ISR = Register(int0=0, int1=0, int2=0, int3=0, int4=1, int5=0, int6=1, int7=1),
  IER = Register(int0=0, int1=0, int2=0, int3=0, int4=0, int5=0, int6=0, int7=0),
  SOFTR = Register(RKEY=write-only),
  CR = Register(EN=0, TX_FIFO_Reset=0, MSMS=0, TX=0, TXAK=0, RSTA=0, GC_EN=0),
  SR = Register(ABGC=0, AAS=0, BB=0, ARW=0, TX_FIFO_Full=0, RX_FIFO_Full=0, RX_FIFO_Empty=1, TX_FIFO_Empty=1),
  TX_FIFO = Register(D7_D0=write-only, Start=write-only, Stop=write-only),
  RX_FIFO = Register(D7_D0=0),
  ADR = Register(Slave_Address=0),
  TX_FIFO_OCY = Register(Occupancy_Value=0),
  RX_FIFO_OCY = Register(Occupancy_Value=0),
  TEN_ADR = Register(MSB_of_Slave_Address=0),
  RX_FIFO_PIRQ = Register(Compare_Value=0),
  GPO = Register(General_Purpose_Outputs=0),
  TSUSTA = Register(TSUSTA=570),
  TSUSTO = Register(TSUSTO=500),
  THDSTA = Register(THDSTA=430),
  TSUDAT = Register(TSUDAT=55),
  TBUF = Register(TBUF=500),
  THIGH = Register(THIGH=493),
  TLOW = Regi

In [61]:
print(hex(iic.read(0x104)))

0xc0


In [60]:
iic.write(0x040,0xA)

In [34]:
rx_data = list(c_data)

In [35]:
rx_data

[107, 101, 3, 98, 198, 221, 33, 64, 76, 47, 201, 85, 215, 144, 107, 255]

In [36]:
rx_byte = bytearray(rx_data)
print( rx_byte.hex())

6b650362c6dd21404c2fc955d7906bff


In [46]:
m="AC0A7F8C2FAAC49775A616B7C0CC21D8"

In [539]:
k="0123456789ABCDEF0011223344556677"

In [540]:
mdata = bytearray.fromhex(m); mdata = list(mdata); mdata

[172, 10, 127, 140, 47, 170, 196, 151, 117, 166, 22, 183, 192, 204, 33, 216]

In [541]:
kdata = bytearray.fromhex(k); kdata = list(kdata); kdata

[1, 35, 69, 103, 137, 171, 205, 239, 0, 17, 34, 51, 68, 85, 102, 119]

In [542]:
i=0
while i<len(mdata):
    print(mdata[i] ^ kdata[i])
    i=i+1

173
41
58
235
166
1
9
120
117
183
52
132
132
153
71
175


In [24]:
test_vector

'00C00010CC0A7F8C2FAAC49775A616B7C0CC21D800C100100123456789ABCDEF00112233445566770081001000800001'

In [30]:
testV = [int(test_vector[i:i+8],16) for i in range(0, len(test_vector), 8)]

In [31]:
testV

[12582928,
 3423240076,
 799720599,
 1973819063,
 3234603480,
 12648464,
 19088743,
 2309737967,
 1122867,
 1146447479,
 8454160,
 8388609]

In [64]:
dutctrl.write(dut_reset,1); dutctrl.write(dut_reset,0)